In [0]:
%pip install google_play_scraper

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
from google_play_scraper import Sort, reviews 
from pyspark.sql.types import StructType, StructField, StringType, DateType, FloatType

In [0]:
%pip install google_play_scraper

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
# Databricks notebook source
# COMMAND ----------

# 1. Instalación de la librería
# MAGIC %pip install google_play_scraper

# COMMAND ----------

# Reinicio del entorno de Python para aplicar la instalación
#dbutils.library.restartPython()

# COMMAND ----------

# 2. Importación de librerías
import pandas as pd
# CAMBIO IMPORTANTE: Importamos 'reviews' en lugar de 'reviews_all'
from google_play_scraper import Sort, reviews 
from pyspark.sql.types import StructType, StructField, StringType, DateType, FloatType

# COMMAND ----------

# 3. Diccionario con los IDs exactos de PlayStore para las apps de transporte
apps_transporte = {
    'Yango': 'com.yandex.yango',
    'inDrive': 'sinet.startup.inDriver',
    'Uber': 'com.ubercab',
    'DiDi': 'com.didiglobal.passenger', 
    'Cabify': 'com.cabify.rider',
    'Uber Lite': 'com.ubercab.uberlite',
    'Bolt': 'ee.mtakso.client',
    'Lyft': 'me.lyft.android',
    'Grab Driver': 'com.grabtaxi.driver2',
    'Yassir': 'com.yatechnologies.yassir_rider',
    'Maxim': 'com.taxsee.taxsee',
    'TaxiF': 'com.taxif.passenger'
}

# COMMAND ----------

# 4. Proceso de Web Scraping Iterativo y Limitado
app_dfs = []

# DEFINIR EL LÍMITE RAZONABLE AQUÍ
limite_comentarios = 1000000 

for app_name, app_id in apps_transporte.items():
    print(f"Extrayendo un máximo de {limite_comentarios} reviews de {app_name} ({app_id})...")
    try:
        # CAMBIO IMPORTANTE: Usamos 'reviews' y pasamos el parámetro 'count'
        # La función devuelve una tupla (resultado, token_de_continuacion). Solo nos interesa el resultado.
        result, _ = reviews(
            app_id,
            lang='es',
            country='BO', 
            sort=Sort.MOST_RELEVANT,
            count=limite_comentarios # Aquí aplicamos el límite
        )
        
        # Validar que la app tenga comentarios
        if result:
            df_temp = pd.json_normalize(result)
            df_temp['App'] = app_name 
            app_dfs.append(df_temp)
            print(f" -> {len(result)} reviews obtenidas con éxito.")
        else:
            print(" -> No se encontraron reviews (probablemente no disponible en la región).")
            
    except Exception as e:
        print(f" -> Error al extraer {app_name}: {e}")

# 5. Transformación y Limpieza (Pandas)
if len(app_dfs) > 0:
    playstore = pd.concat(app_dfs, ignore_index=True)
    
    # Filtrar columnas requeridas
    playstore = playstore[['reviewId', 'at', 'score', 'reviewCreatedVersion', 'content', 'App']]
    playstore['Store'] = 'PlayStore'
    
    # Renombrar para coincidir con el esquema deseado
    playstore = playstore.rename(
        columns={
            'reviewId': 'id',
            'at': 'date',
            'score': 'rating',
            'reviewCreatedVersion': 'version'
        }
    )
    
    # Homologar la columna de fecha para evitar errores en PySpark
    playstore['date'] = pd.to_datetime(playstore['date']).dt.date
    
    # Mostrar una vista previa rápida
    print("Transformación completada. Mostrando los primeros registros:")
    display(playstore.head())

    # 6. Conversión a Spark DataFrame y escritura en Delta Table
    schema = StructType([
        StructField("id", StringType(), True),
        StructField("date", DateType(), True),
        StructField("rating", FloatType(), True),
        StructField("version", StringType(), True),
        StructField("content", StringType(), True),
        StructField("App", StringType(), True),
        StructField("Store", StringType(), True)
    ])

    # Crear DataFrame en Spark
    df = spark.createDataFrame(playstore, schema=schema)

    # Guardar como tabla Delta
    df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.transport_apps_reviews2")
    print("¡Tabla guardada con éxito en workspace.default.transport_apps_reviews!")
    
else:
    print("Error: No se obtuvieron datos en el Paso 4. Revisa si la ejecución anterior arrojó algún error.")

Extrayendo un máximo de 1000000 reviews de Yango (com.yandex.yango)...
 -> 62924 reviews obtenidas con éxito.
Extrayendo un máximo de 1000000 reviews de inDrive (sinet.startup.inDriver)...
 -> 513033 reviews obtenidas con éxito.
Extrayendo un máximo de 1000000 reviews de Uber (com.ubercab)...
 -> 310500 reviews obtenidas con éxito.
Extrayendo un máximo de 1000000 reviews de DiDi (com.didiglobal.passenger)...
 -> 337500 reviews obtenidas con éxito.
Extrayendo un máximo de 1000000 reviews de Cabify (com.cabify.rider)...
 -> 113319 reviews obtenidas con éxito.
Extrayendo un máximo de 1000000 reviews de Uber Lite (com.ubercab.uberlite)...
 -> 41691 reviews obtenidas con éxito.
Extrayendo un máximo de 1000000 reviews de Bolt (ee.mtakso.client)...
 -> 50477 reviews obtenidas con éxito.
Extrayendo un máximo de 1000000 reviews de Lyft (me.lyft.android)...
 -> 8202 reviews obtenidas con éxito.
Extrayendo un máximo de 1000000 reviews de Grab Driver (com.grabtaxi.driver2)...
 -> 49 reviews obteni

id,date,rating,version,content,App,Store
3f0ad9e2-b5da-4112-aeaf-7b7f9aeadaee,2026-06-02,4,5.79.0,"Es una buena app para transporte. Pero en mi opinión le doy solo 4 estrellas porque considero que deberían mejorar la precisión del mapa. Uno solicita el servicio y a veces se va a otra dirección, y con lo que uno está apurado creemos que se marco bien, pero no, generalmente pasa en el momento de recojo. Está situación en muchas ocasiones ha generado quejas del usuario al chófer o viceversa. Espero que la app tome en cuenta este detalle.",Yango,PlayStore
936a0c5e-40e4-49e2-91d2-1bbf59739b26,2026-06-03,1,5.79.0,"ya una vez use yango envios y no llego el vehículo que solicité, después de tiempo volví a instalar la app y pedí un viaje agregue pago con yape, cuando el conductor llegó a insistencia me quería hacer pagar en efectivo o que le yapeara a el directo, y menos mal no puse con tarjeta por qué me dijo si es con tarjeta que mejor YO cancele y no suba, aparte de eso el chófer es muy lisuriento toda la ruta diciendo sandeces. mal mal servicio",Yango,PlayStore
3e81d534-b3f6-43d4-a72d-6fc36e2549e3,2026-05-30,1,5.79.0,"Siguen teniendo graves problemas con la georeferencia, además de no ubicar al conductor en la dirección correcta, sus tiempos de espera tampoco corresponden. Dicen 5 minutos que terminan siendo mínimo 10. Los conductores excelentes, pero el problema es la plataforma",Yango,PlayStore
301748b0-7ea7-4d53-a8e2-d162c929abc9,2026-06-01,1,5.79.0,PUBLICIDAD ENGAÑOSA Y CONFUSA. Cómo es posible que a mi me aparezca equis valor con descuento y que al motorizado le aparezca ese mismo valor y además pueda escoger entre ese valor o el que aparece sin descuento. CLARAMENTE todos los motociclistas van a escoger el valor más alto y no es culpa de ellos. Es culpa de la publicidad engañosa que está app le da a los usuarios pasajeros. FALSOS Y DENUNCIABLE ANTE LA SIC. Los precios deben ser exactos.,Yango,PlayStore
c63984ab-94db-4660-82bf-de7ef55ee9cf,2026-05-09,1,5.73.1,"mi experiencia con esta App es de cero estrellas, es una aplicación deficiente, nunca tiene cobertura, cuando la intento usar me sale un servicio económico pero me obligan a seleccionar uno costoso, no importa donde esté siempre me dice que la dirección de destino u origen no tiene cobertura en el mapa pero si ne arroja los costos, es muy lenta, curiosamente las otras aplicaciones de transporte corren bien en mi teléfono excepto esta, en fin, mejoren su sistema.",Yango,PlayStore


¡Tabla guardada con éxito en workspace.default.transport_apps_reviews!


In [0]:
df.summary()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8549364686978109>, line 1
----> 1 df.summary()

NameError: name 'df' is not defined

In [0]:
# COMMAND ----------

# 5 (Continuación). Terminamos de homologar la fecha
playstore['date'] = pd.to_datetime(playstore['date']).dt.date

display(playstore.head())

# COMMAND ----------

# 6. Conversión a Spark DataFrame y escritura en Delta Table
schema = StructType([
    StructField("id", StringType(), True),
    StructField("date", DateType(), True),
    StructField("rating", FloatType(), True),
    StructField("version", StringType(), True),
    StructField("content", StringType(), True),
    StructField("App", StringType(), True),
    StructField("Store", StringType(), True)
])

# Crear DataFrame en Spark
df = spark.createDataFrame(playstore, schema=schema)

# Guardar como tabla Delta
df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.transport_apps_reviews")
print("¡Tabla guardada con éxito!")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8549364686978103>, line 4
      1 # COMMAND ----------
      2 
      3 # 5 (Continuación). Terminamos de homologar la fecha
----> 4 playstore['date'] = pd.to_datetime(playstore['date']).dt.date
      6 display(playstore.head())
      8 # COMMAND ----------
      9 
     10 # 6. Conversión a Spark DataFrame y escritura en Delta Table

NameError: name 'pd' is not defined

In [0]:
%sql
SELECT * FROM workspace.default.transport_apps_reviews LIMIT 10;

id,date,rating,version,content,App,Store
f5deb691-8a6c-4f0d-b0ec-e7d07949f414,2025-07-02,1.0,null,"Cuando me fije en casa, si me aparecia la dirección del cine, pero al salir de noche del mismo, no me aparecia en la app dicha ubicación y me volví en otro medio de transporte. Malísima la app hasta donde yo se.",Uber Lite,PlayStore
2655fd0a-f612-48dc-aa96-5661eaabeff6,2019-08-18,1.0,1.55.10001,"Me han cobrado doble en más de una ocasión y jamás recibí el reembolso. La aplicación no me permite elegir en el mapa mi destino, lo que es un enorme inconveniente.",Uber Lite,PlayStore
965b910b-f4e1-4fb1-9659-f863e3cdaa04,2019-11-30,5.0,1.78.10001,"Aveces descontrolan a uno porque tomo el servicio y me aparece por tal costo$$, y resulta q al llegar a mi destino ya es otra cantidad y eso no me parece y no es porque tenga algún pendiente con ustedes , y así no conviene. Ojalá tomen en cuenta . Este comentario. Gracias.",Uber Lite,PlayStore
f9f9e85c-593b-44d8-b175-a97cf9822fd5,2019-11-28,3.0,1.77.10001,"La verdad es muy buena la app, ya que funcióna bien y ocupa poco espacio, pero hace días hice pago con tarjeta débito y me hicieron un cobro además del cobro del viaje y se suponía que me lo iban a reémbolsar y hasta ahorita no lo han hecho y no es la primera vez que pasa, de ahí en más todo bien",Uber Lite,PlayStore
91c4bcb3-4511-466b-9d5b-5d986bd6dfac,2019-06-19,4.0,1.52.10000,"Buena app. Permite solicitar un taxi de manera sencilla y básica. PERO deben corregir el error de que no permite contactar ni al usuario ni al conductor, a ambos envía hacia una llamada a un número de teléfono fijo, mas no al móvil del usuario ni del conductor.",Uber Lite,PlayStore
54fd2182-5f5a-4e51-8fdd-5941376386a0,2020-02-22,1.0,1.94.10000,"No deja volver a iniciar sesión si cambia de teléfono o algo así por el estilo por qué solo aparece la opción de comenzar que es para registrarse, y como uno ya se ha registrado le pone problema y no hay ninguna opción de iniciar sesión de nuevo. Por eso ya no uso Uber!!!",Uber Lite,PlayStore
eb7a318e-5c01-4cac-b699-41531a65d51f,2020-03-02,1.0,null,El vehículo se queda fijo en el punto de partida y no avanza por lo tanto no se puede saber que tan cerca viene. Además de que marca una posición errónea de partida sin poder modificarla.,Uber Lite,PlayStore
96137389-2978-4445-b4a4-2dbc0cf667b2,2023-05-10,1.0,1.149.10000,"ya no funciona como antes cuando me va a dar el precio del viaje se cierra la app, además de que tarda mucho en conectarse o luego no conecta a pesar de tener buen internet es una lástima ya que funcionaba muy bien",Uber Lite,PlayStore
fef34610-0c69-4839-9927-2f92eb72ea16,2020-08-03,2.0,1.110.10000,"En los últimos 4 días he tenido muchos problemas con la aplicación. Después de pedir un servicio, la aplicación no me deja ver el chófer, placas ni precios. Me saca del servicio como si fuera a pedir uno nuevo. Y llega el chófer pero en mi aplicación no aparece la solicitud.",Uber Lite,PlayStore
1f34d14b-bcda-4e24-8be0-61c8cc4bb964,2020-09-01,3.0,1.110.10000,"La aplicación esta bien pero el servicio de uber la verdad cada vez es mas malo, hay conductores que manejan horrible, los precios cada vez son mas caros y me cobran minutos de mas cuando estoy arriba del vehiculo",Uber Lite,PlayStore
